In [ ]:
from functools import partial

import matplotlib.pyplot as plt
import numpy as np

import meow as mw

## 1. Define the rib waveguide geometry

In [ ]:
# Parameters (matching emepy_diverging)
wl = 1.0
widths = [0.8, 1.0]
t_slab = 0.2
t_soi = 1.5
t_ox = 0.2
n_Si = 3.4
n_SiO2 = 1.4
w_sim = 3.0
h_sim = 2.5
y_bot = -0.5

si = mw.IndexMaterial(n=n_Si, name="Si", meta={"color": "orange"})
sio2 = mw.IndexMaterial(n=n_SiO2, name="SiO2", meta={"color": "steelblue"})

env = mw.Environment(wl=wl, T=25.0)
mesh = mw.Mesh2D(
    x=np.linspace(-w_sim / 2, w_sim / 2, 101),
    y=np.linspace(y_bot, y_bot + h_sim, 101),
    num_pml=(10, 10),
)


def make_cross_section(w):
    """Build a CrossSection for a rib waveguide of width w."""
    structures = [
        # SiO2 base layer (below slab + oxide)
        mw.Structure2D(
            material=sio2,
            geometry=mw.Rectangle(
                x_min=-w_sim / 2,
                x_max=w_sim / 2,
                y_min=y_bot,
                y_max=t_slab + t_ox,
            ),
            mesh_order=2,
        ),
        # SiO2 cladding around ridge
        mw.Structure2D(
            material=sio2,
            geometry=mw.Rectangle(
                x_min=-w / 2 - t_ox / 2,
                x_max=w / 2 + t_ox / 2,
                y_min=t_slab + t_ox,
                y_max=t_soi + t_ox,
            ),
            mesh_order=2,
        ),
        # Si slab (full width)
        mw.Structure2D(
            material=si,
            geometry=mw.Rectangle(
                x_min=-w_sim / 2,
                x_max=w_sim / 2,
                y_min=0.0,
                y_max=t_slab,
            ),
            mesh_order=1,
        ),
        # Si ridge
        mw.Structure2D(
            material=si,
            geometry=mw.Rectangle(
                x_min=-w / 2,
                x_max=w / 2,
                y_min=t_slab,
                y_max=t_soi,
            ),
            mesh_order=1,
        ),
    ]
    return mw.CrossSection(structures=structures, mesh=mesh, env=env)


cs_left = make_cross_section(widths[0])
cs_right = make_cross_section(widths[1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cs_left._visualize(ax=axes[0], show=False)
axes[0].set_title(f"Left (w={widths[0]} um)")
cs_right._visualize(ax=axes[1], show=False)
axes[1].set_title(f"Right (w={widths[1]} um)")
plt.tight_layout()
plt.show()

## 2. Compute and visualize modes for both cross-sections

In [ ]:
# Compute modes for both cross-sections
num_modes = 100

inner_product = partial(
    mw.inner_product,
    symmetric=True,
    conjugate=True,
    ignore_pml=False,
    interpolation=None,
)

modes_left = mw.compute_modes(cs_left, num_modes=num_modes, inner_product=inner_product)
modes_right = mw.compute_modes(
    cs_right, num_modes=num_modes, inner_product=inner_product
)

print(f"Computed {len(modes_left)} left/{len(modes_right)} right raw modes; ")  # fmt: skip

modes_left = mw.orthonormalize(modes_left, inner_product)
modes_right = mw.orthonormalize(modes_right, inner_product)

print(f"kept {len(modes_left)} left/{len(modes_right)} right after orthonormalization")  # fmt: skip

In [ ]:
# Visualize left and right mode sets
print("Left cross-section modes")
mw.visualize(modes_left[:2], fields=("Ex", "Hx"))

print("Right cross-section modes")
mw.visualize(modes_right[:2], fields=("Ex", "Hx"))

In [ ]:
def overlap_matrix(modes, op):
    n = len(modes)
    M = np.zeros((n, n), dtype=np.complex128)
    for i, mi in enumerate(modes):
        for j, mj in enumerate(modes):
            M[i, j] = op(mi, mj)
    return M


def report_orthonormality(name, modes, inner_product):
    I = np.eye(len(modes), dtype=np.complex128)
    M_dot = overlap_matrix(modes, inner_product)
    M_dot_conj = overlap_matrix(modes, partial(inner_product, conjugate=True))

    err_dot = np.linalg.norm(M_dot - I)
    err_dot_conj = np.linalg.norm(M_dot_conj - I)

    print(f"\n{name}")
    print(f"||dot - I||_F      = {err_dot:.3e}")
    print(f"||dot_conj - I||_F = {err_dot_conj:.3e}")

    if err_dot < err_dot_conj:
        print("Orthonormal for: dot")
    elif err_dot_conj < err_dot:
        print("Orthonormal for: dot_conj")
    else:
        print("Orthonormal for: neither (or tie)")

    return M_dot, M_dot_conj


M_left_dot, M_left_dot_conj = report_orthonormality("Left cross-section", modes_left, inner_product)  # fmt: skip
M_right_dot, M_right_dot_conj = report_orthonormality("Right cross-section", modes_right, inner_product)  # fmt: skip

## 3. Interface Matrices (Page 73)

In [ ]:
# Page-73 interface matrices (thesis)
# O_lr(i,j) = <E_i^l, H_j^r>
# O_rl(i,j) = <E_i^r, H_j^l>
# T_lr = 2 (O_lr + O_rl^T)^(-1)
# R_lr = 1/2 (O_rl^T - O_lr) T_lr


def overlap_cross(modes_a, modes_b, op):
    M = np.zeros((len(modes_a), len(modes_b)), dtype=np.complex128)
    for i, mi in enumerate(modes_a):
        for j, mj in enumerate(modes_b):
            M[i, j] = op(mi, mj)
    return M


def hermitianize(M):
    return 0.5 * (M + M.conj().T)


def regularize_metric(G, rel_floor=1e-10, abs_floor=1e-12):
    Gh = hermitianize(G)
    w, V = np.linalg.eigh(Gh)
    floor = max(abs_floor, rel_floor * max(1.0, float(np.max(np.abs(w)))))
    w_clip = np.maximum(w.real, floor)
    G_reg = (V * w_clip) @ V.conj().T
    C = np.diag(np.sqrt(w_clip)) @ V.conj().T
    C_inv = V @ np.diag(1.0 / np.sqrt(w_clip))
    return G_reg, C, C_inv, w, w_clip, floor


def metric_passivity_residual(T, G_in, G_out):
    M = hermitianize(T.conj().T @ G_out @ T - G_in)
    return float(np.max(np.linalg.eigvalsh(M).real))


# Use a moderate subspace for a stable first experiment.
N_interface = 100  # min(30, len(modes_left), len(modes_right))
mL = modes_left[:N_interface]
mR = modes_right[:N_interface]

O_lr = overlap_cross(mL, mR, inner_product)
O_rl = overlap_cross(mR, mL, inner_product)

S = O_lr + O_rl.T
S_inv = np.linalg.pinv(S)
T_lr = 2.0 * S_inv
R_lr = 0.5 * (O_rl.T - O_lr) @ T_lr

# Metric extraction (always) using the same overlap product as interface matching.
G_LL = overlap_cross(mL, mL, inner_product)
G_RR = overlap_cross(mR, mR, inner_product)
G_LL_reg, C_L, C_L_inv, wL, wL_clip, floorL = regularize_metric(G_LL)
G_RR_reg, C_R, C_R_inv, wR, wR_clip, floorR = regularize_metric(G_RR)

# Metric-aware passivity projection: whiten -> clip -> unwhiten.
T_tilde = C_R @ T_lr @ C_L_inv
U, s, Vh = np.linalg.svd(T_tilde, full_matrices=False)
s_clip = np.minimum(s, 1.0)
T_tilde_proj = (U * s_clip) @ Vh
T_lr_proj = C_R_inv @ T_tilde_proj @ C_L
R_lr_proj = 0.5 * (O_rl.T - O_lr) @ T_lr_proj

I = np.eye(N_interface, dtype=np.complex128)

print(f"N_interface = {N_interface}")
print(f"cond(O_lr + O_rl^T) = {np.linalg.cond(S):.3e}")
print(f"||(O_lr + O_rl^T) T_lr - 2I||_F = {np.linalg.norm(S @ T_lr - 2 * I):.3e}")
print(f"||R_lr||_F = {np.linalg.norm(R_lr):.3e}")
print(f"metric floors: left={floorL:.3e}, right={floorR:.3e}")

res_raw = metric_passivity_residual(T_lr, G_LL_reg, G_RR_reg)
res_proj = metric_passivity_residual(T_lr_proj, G_LL_reg, G_RR_reg)
print(f"passivity residual raw  lambda_max(T^H G_RR T - G_LL) = {res_raw:.3e}")
print(f"passivity residual proj lambda_max(T^H G_RR T - G_LL) = {res_proj:.3e}")

print(f"max sigma raw/proj in whitened space: {np.max(s):.6f} / {np.max(s_clip):.6f}")
print(f"||T_lr_proj - T_lr||_F = {np.linalg.norm(T_lr_proj - T_lr):.3e}")

# Amplitude mapping example for unit excitation of mode 0 from the left.
a_inc = np.zeros(N_interface, dtype=np.complex128)
a_inc[0] = 1.0

a_refl = R_lr @ a_inc
a_trans = T_lr @ a_inc
a_refl_proj = R_lr_proj @ a_inc
a_trans_proj = T_lr_proj @ a_inc

print(
    f"||a_refl||_2 raw/proj = {np.linalg.norm(a_refl):.3e} / {np.linalg.norm(a_refl_proj):.3e}"
)
print(
    f"||a_trans||_2 raw/proj = {np.linalg.norm(a_trans):.3e} / {np.linalg.norm(a_trans_proj):.3e}"
)

In [ ]:
# Same-medium sanity check (left -> left)
O_ll = overlap_cross(mL, mL, inner_product)
S_ll = O_ll + O_ll.T
S_ll_inv = np.linalg.pinv(S_ll)
T_ll = 2.0 * S_ll_inv
R_ll = 0.5 * (O_ll.T - O_ll) @ T_ll

print("same-medium sanity (using left basis on both sides)")
print(f"||T_ll - I||_F = {np.linalg.norm(T_ll - I):.3e}")
print(f"||R_ll||_F     = {np.linalg.norm(R_ll):.3e}")